In [1]:
import pymongo
import pandas
import sqlite3

## Connexion à la base en SQLite

In [ ]:
conn = sqlite3.connect("world.sqlite")

In [ ]:
pandas.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,City
1,Country
2,CountryLanguage


On récupère les données des 3 tables présentes.

In [20]:
liste_tables = pandas.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)["name"]
for table in liste_tables:
    valeurs = pandas.read_sql_query("SELECT * FROM " + table + ";", conn)
    globals()[table] = valeurs

# on ferme la connexion à la base SQLite car ce n'est plus utile
conn.close()

In [21]:
City.head()

,ID,Name,CountryCode,District,Population
0,1,Kabul,AFG,Kabol,1780000
1,2,Qandahar,AFG,Qandahar,237500
2,3,Herat,AFG,Herat,186800
3,4,Mazar-e-Sharif,AFG,Balkh,127800
4,5,Amsterdam,NLD,Noord-Holland,731200


In [22]:
Country.head()

,Code,Name,Continent,Region,SurfaceArea,IndepYear,Population,LifeExpectancy,GNP,GNPOld,LocalName,GovernmentForm,HeadOfState,Capital,Code2
0,ABW,Aruba,North America,Caribbean,193.0,NaN,103000,78.4,828.0,793.0,Aruba,Nonmetropolitan Territory of The Netherlands,Beatrix,129.0,AW
1,AFG,Afghanistan,Asia,Southern and Central Asia,652090.0,1919.0,22720000,45.9,5976.0,NaN,Afganistan/Afqanestan,Islamic Emirate,Mohammad Omar,1.0,AF
2,AGO,Angola,Africa,Central Africa,1246700.0,1975.0,12878000,38.3,6648.0,7984.0,Angola,Republic,José Eduardo dos Santos,56.0,AO
3,AIA,Anguilla,North America,Caribbean,96.0,NaN,8000,76.1,63.2,NaN,Anguilla,Dependent Territory of the UK,Elisabeth II,62.0,AI
4,ALB,Albania,Europe,Southern Europe,28748.0,1912.0,3401200,71.6,3205.0,2500.0,Shqipëria,Republic,Rexhep Mejdani,34.0,AL


In [23]:
CountryLanguage.head()

,CountryCode,Language,IsOfficial,Percentage
0,ABW,Dutch,T,5.3
1,ABW,English,F,9.5
2,ABW,Papiamento,F,76.7
3,ABW,Spanish,F,7.4
4,AFG,Balochi,F,0.9


## Connexion à Mongo DB

In [24]:
client = pymongo.MongoClient()

# On va créer une nouvelle base de données nommée "world"
db = client.world

## Première possibilité : on fait une collection par table

In [52]:
[dict(zip(list(City), e)) for e in City.values.tolist()]

[{'ID': 1,
  'Name': 'Kabul',
  'CountryCode': 'AFG',
  'District': 'Kabol',
  'Population': 1780000},
 {'ID': 2,
  'Name': 'Qandahar',
  'CountryCode': 'AFG',
  'District': 'Qandahar',
  'Population': 237500},
 {'ID': 3,
  'Name': 'Herat',
  'CountryCode': 'AFG',
  'District': 'Herat',
  'Population': 186800},
 {'ID': 4,
  'Name': 'Mazar-e-Sharif',
  'CountryCode': 'AFG',
  'District': 'Balkh',
  'Population': 127800},
 {'ID': 5,
  'Name': 'Amsterdam',
  'CountryCode': 'NLD',
  'District': 'Noord-Holland',
  'Population': 731200},
 {'ID': 6,
  'Name': 'Rotterdam',
  'CountryCode': 'NLD',
  'District': 'Zuid-Holland',
  'Population': 593321},
 {'ID': 7,
  'Name': 'Haag',
  'CountryCode': 'NLD',
  'District': 'Zuid-Holland',
  'Population': 440900},
 {'ID': 8,
  'Name': 'Utrecht',
  'CountryCode': 'NLD',
  'District': 'Utrecht',
  'Population': 234323},
 {'ID': 9,
  'Name': 'Eindhoven',
  'CountryCode': 'NLD',
  'District': 'Noord-Brabant',
  'Population': 201843},
 {'ID': 10,
  'Name':

In [58]:
for table in liste_tables:
    db[table].insert_many([dict(zip(list(globals()[table]), e)) for e in globals()[table].values.tolist()])

In [ ]:
conn.close()

In [64]:
# Liste des villes de France
res = db.City.aggregate([
    { "$lookup": {
        "from": "Country",
        "localField": "CountryCode",
        "foreignField": "Code",
        "as": "Country"
    }},
    { "$match": { "Country.Name": "France" }}
])
pandas.DataFrame(res)

,_id,ID,Name,CountryCode,District,Population,Country
0,68d29ff1987faa03afc0efe6,2974,Paris,FRA,Île-de-France,2125246,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."
1,68d29ff1987faa03afc0efe7,2975,Marseille,FRA,Provence-Alpes-Côte,798430,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."
2,68d29ff1987faa03afc0efe8,2976,Lyon,FRA,Rhône-Alpes,445452,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."
3,68d29ff1987faa03afc0efe9,2977,Toulouse,FRA,Midi-Pyrénées,390350,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."
4,68d29ff1987faa03afc0efea,2978,Nice,FRA,Provence-Alpes-Côte,342738,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."
5,68d29ff1987faa03afc0efeb,2979,Nantes,FRA,Pays de la Loire,270251,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."
6,68d29ff1987faa03afc0efec,2980,Strasbourg,FRA,Alsace,264115,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."
7,68d29ff1987faa03afc0efed,2981,Montpellier,FRA,Languedoc-Roussillon,225392,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."
8,68d29ff1987faa03afc0efee,2982,Bordeaux,FRA,Aquitaine,215363,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."
9,68d29ff1987faa03afc0efef,2983,Rennes,FRA,Haute-Normandie,206229,"[{'_id': 68d29ff1987faa03afc0f480, 'Code': 'FR..."


## Deuxième possibilité : une collection avec toutes les informations

In [76]:
liste_cities = [City.query("CountryCode == @e") for e in Country.Code]

In [81]:
liste_cities_propre = [[dict(zip(list(l), e)) for e in l.values.tolist()] for l in liste_cities]

In [85]:
Country_new = [dict(zip(list(Country), e)) for e in Country.values.tolist()]
Country_new

[{'Code': 'ABW',
  'Name': 'Aruba',
  'Continent': 'North America',
  'Region': 'Caribbean',
  'SurfaceArea': 193.0,
  'IndepYear': nan,
  'Population': 103000,
  'LifeExpectancy': 78.4,
  'GNP': 828.0,
  'GNPOld': 793.0,
  'LocalName': 'Aruba',
  'GovernmentForm': 'Nonmetropolitan Territory of The Netherlands',
  'HeadOfState': 'Beatrix',
  'Capital': 129.0,
  'Code2': 'AW'},
 {'Code': 'AFG',
  'Name': 'Afghanistan',
  'Continent': 'Asia',
  'Region': 'Southern and Central Asia',
  'SurfaceArea': 652090.0,
  'IndepYear': 1919.0,
  'Population': 22720000,
  'LifeExpectancy': 45.9,
  'GNP': 5976.0,
  'GNPOld': nan,
  'LocalName': 'Afganistan/Afqanestan',
  'GovernmentForm': 'Islamic Emirate',
  'HeadOfState': 'Mohammad Omar',
  'Capital': 1.0,
  'Code2': 'AF'},
 {'Code': 'AGO',
  'Name': 'Angola',
  'Continent': 'Africa',
  'Region': 'Central Africa',
  'SurfaceArea': 1246700.0,
  'IndepYear': 1975.0,
  'Population': 12878000,
  'LifeExpectancy': 38.3,
  'GNP': 6648.0,
  'GNPOld': 7984.

In [99]:
for e in Country_new:
    pays = e["Code"]
    liste_villes = City.query("CountryCode == @pays")
    liste_propre = [dict(zip(list(liste_villes), e)) for e in liste_villes.values.tolist()]
    e["Cities"] = liste_propre
    liste_langues = CountryLanguage.query("CountryCode == @pays")
    liste_propre2 = [dict(zip(list(liste_langues), e)) for e in liste_langues.values.tolist()]
    e["Langues"] = liste_propre2
Country_new[1]

{'Code': 'AFG',
 'Name': 'Afghanistan',
 'Continent': 'Asia',
 'Region': 'Southern and Central Asia',
 'SurfaceArea': 652090.0,
 'IndepYear': 1919.0,
 'Population': 22720000,
 'LifeExpectancy': 45.9,
 'GNP': 5976.0,
 'GNPOld': nan,
 'LocalName': 'Afganistan/Afqanestan',
 'GovernmentForm': 'Islamic Emirate',
 'HeadOfState': 'Mohammad Omar',
 'Capital': 1.0,
 'Code2': 'AF',
 'Cities': [{'ID': 1,
   'Name': 'Kabul',
   'CountryCode': 'AFG',
   'District': 'Kabol',
   'Population': 1780000},
  {'ID': 2,
   'Name': 'Qandahar',
   'CountryCode': 'AFG',
   'District': 'Qandahar',
   'Population': 237500},
  {'ID': 3,
   'Name': 'Herat',
   'CountryCode': 'AFG',
   'District': 'Herat',
   'Population': 186800},
  {'ID': 4,
   'Name': 'Mazar-e-Sharif',
   'CountryCode': 'AFG',
   'District': 'Balkh',
   'Population': 127800}],
 'Langues': [{'CountryCode': 'AFG',
   'Language': 'Balochi',
   'IsOfficial': 'F',
   'Percentage': 0.9},
  {'CountryCode': 'AFG',
   'Language': 'Dari',
   'IsOfficial'

In [101]:
db["CountryALL"].insert_many(Country_new)

InsertManyResult([ObjectId('68d2a59d987faa03afc0f8ff'), ObjectId('68d2a59d987faa03afc0f900'), ObjectId('68d2a59d987faa03afc0f901'), ObjectId('68d2a59d987faa03afc0f902'), ObjectId('68d2a59d987faa03afc0f903'), ObjectId('68d2a59d987faa03afc0f904'), ObjectId('68d2a59d987faa03afc0f905'), ObjectId('68d2a59d987faa03afc0f906'), ObjectId('68d2a59d987faa03afc0f907'), ObjectId('68d2a59d987faa03afc0f908'), ObjectId('68d2a59d987faa03afc0f909'), ObjectId('68d2a59d987faa03afc0f90a'), ObjectId('68d2a59d987faa03afc0f90b'), ObjectId('68d2a59d987faa03afc0f90c'), ObjectId('68d2a59d987faa03afc0f90d'), ObjectId('68d2a59d987faa03afc0f90e'), ObjectId('68d2a59d987faa03afc0f90f'), ObjectId('68d2a59d987faa03afc0f910'), ObjectId('68d2a59d987faa03afc0f911'), ObjectId('68d2a59d987faa03afc0f912'), ObjectId('68d2a59d987faa03afc0f913'), ObjectId('68d2a59d987faa03afc0f914'), ObjectId('68d2a59d987faa03afc0f915'), ObjectId('68d2a59d987faa03afc0f916'), ObjectId('68d2a59d987faa03afc0f917'), ObjectId('68d2a59d987faa03afc0f9

In [105]:
# Liste des villes de France
res = db.CountryALL.aggregate([
    { "$match": { "Name": "France" }},
    { "$unwind": "$Cities" },
    { "$project": { "_id": 0, "Ville": "$Cities.Name" }}
])
pandas.DataFrame(res)

,Ville
0,Paris
1,Marseille
2,Lyon
3,Toulouse
4,Nice
5,Nantes
6,Strasbourg
7,Montpellier
8,Bordeaux
9,Rennes
